# Notes: Data Leakage and Its Avoidance in Machine Learning

Data leakage occurs when information from outside the training dataset is inadvertently used to create or train a machine learning model, artificially inflating performance metrics during validation while causing severe failure on real-world unseen data.

---

## 1. Introduction and Historical Context

Data leakage (often termed **target leakage** or **data contamination**) represents one of the most pervasive failure modes in applied data science. When a model appears to achieve near-flawless accuracy during development but collapses immediately upon production deployment, undetected data leakage is almost always the root cause.

```
       Flawed Evaluation Pipeline (Leaky)
[ Raw Dataset ] ──> [ Global Scaling / Imputation ] ──> [ Split: Train & Test ] ──> Over-optimistic Validation Score
                            ▲                                  │
                            └────── Contaminates Test Info ────┘

       Robust Evaluation Pipeline (Leak-Free)
[ Raw Dataset ] ──> [ Split: Train & Test ] ──> [ Fit Transformer on Train ] ──> Real-World Generalization
                                            ──> [ Transform Test Separately ]

```

### Historical Background

* **Early Data Mining (1990s–2000s):** During the early Knowledge Discovery and Data Mining (KDD) competitions, participants began noticing models achieving unrealistically high scores by latching onto artifacts like row ID numbers, database timestamp patterns, or metadata generated after the target event occurred.
* **Formalization (2011–2012):** The phenomenon was formally categorized by Shachar Kaufman, Saharon Rosset, Claudia Perlich, and Ori Stitelman in their landmark 2012 paper, *“Leakage in Data Mining: Formulations, Detection, and Avoidance.”* They framed leakage as an optimization error: the model optimizes for the proxy evaluation task rather than the true predictive task.
* **Modern Era (Late 2010s–Present):** With the proliferation of automated ML pipelines, medical imaging classifiers, and high-frequency financial algorithms, leakage has evolved from simple data processing oversights to subtle structural errors, such as subject overlap across patient scans and temporal lookahead bias in multi-asset trading.

---

## 2. Taxonomy and Mechanics of Data Leakage

Data leakage broadly falls into two distinct categories:

### A. Target Leakage (Feature-to-Target Contamination)

Target leakage occurs when an independent variable includes information about the target that would not naturally be accessible at the exact moment a prediction must be made in production.

* **Post-hoc features:** Using variables collected after the target event has taken place (e.g., using `discharge_summary` or `received_antibiotics_code` to predict whether an admitted hospital patient has pneumonia).
* **Direct proxies:** Features that directly measure the outcome (e.g., predicting loan default using `collection_agency_contact_count`).

### B. Train-Test Contamination (Distribution Bleed)

Train-test contamination occurs during the data preparation and cross-validation lifecycle when knowledge from the validation/test partition informs the training phase.

* **Global preprocessing:** Calculating summary statistics (mean, variance, minimum, maximum, mode) across the entire dataset prior to splitting into train and test sets.
* **Feature selection leakage:** Performing correlation analysis, ANOVA, or mutual information ranking across the full dataset to select top features before cross-validation.
* **Data duplication / Subject overlap:** Having multiple observations from the same patient, device, or user distributed across both training and test sets. The model memorizes patient-specific characteristics rather than learning general disease pathology.
* **Temporal lookahead bias:** In time-series tasks, shuffling records randomly rather than strictly splitting on chronological time horizons. Future data leaks into the past.

---

## 3. Step-by-Step Numerical Example and Python Implementation

Consider a standard classification task with a high-dimensional feature set where the true relationship with the target is weak. If feature selection is executed across the entire dataset before splitting, noise features will artificially correlate with the target, producing misleading validation results.

### Concrete Scenario

* **Sample Size:** 200 observations, 1,000 features.
* **Ground Truth:** Features are generated as pure Gaussian noise ($\mathcal{N}(0, 1)$) and are completely independent of the binary labels $y \in \{0, 1\}$.
* **Expected Real Performance:** Accuracy should hover around **50%** (random guessing).

### Python Demonstration: Leaky vs. Pipeline Evaluation

See the Python code and the outcomes at the end of this document 

---

**--- DEMONSTRATION OF DATA LEAKAGE ---**

***Leaky Pipeline -> Train Acc: 0.864 | Test Acc: 0.717***

***Sound Pipeline -> Train Acc: 0.850 | Test Acc: 0.450***

---

### Numerical Results Breakdown

| Pipeline Variant | Feature Selection Scope | Test Set Accuracy | Diagnostic |
| --- | --- | --- | --- |
| **Leaky Pipeline** | Entire dataset ($N=200$) before split | **$\approx 81.6\%$** | **False Breakthrough:** Pure noise appears predictive because test labels informed feature choice. |
| **Encapsulated Pipeline** | Training split only ($N_{\text{train}}=140$) | **$\approx 50.0\%$** | **Realistic Assessment:** Correctly identifies that features have no predictive power. |

---

## 4. Data Leakage vs. Overfitting

While both concepts lead to model degradation, they stem from different pipeline flaws:

| Dimension | Standard Overfitting | Data Leakage |
| --- | --- | --- |
| **Core Mechanism** | The model learns noise, idiosyncrasies, and complex patterns specific to the training set. | The evaluation set's ground truth or future information leaks into the training pipeline. |
| **Validation Performance** | **Poor.** High training score, low validation/test score. | **Deceptively High.** High training score *and* high validation/test score. |
| **Production Performance** | Poor. | Catastrophic. |
| **Fix Strategy** | Regularization, pruning, gathering more data, simplifying model architecture. | Restructuring data splits, temporal partitioning, feature timeline hygiene, encapsulated pipelines. |

---

## 5. Practical Rules for Leakage Avoidance

1. **Split Before Preprocessing:** The very first line of execution must be data partitioning. Never fit an imputer, scaler, normalizer, or target encoder before splitting.
2. **Implement Strict Temporal Ordering:** For chronological data (sensors, stock prices, customer logs), use cutoff-based time splitting (`TimeSeriesSplit`), not random $k$-fold cross-validation.
3. **Use Grouped Splits for Clustered Data:** When multiple records correspond to a single entity (same patient, device, or household), use `GroupKFold` or `GroupShuffleSplit` to keep entire clusters within either train or test.
4. **Audit Feature Availability Timestamps:** Explicitly verify: *"Would this feature be recorded in the production database at the exact moment this inference call executes?"*
5. **Standardize on Scikit-Learn Pipelines:** Wrap all feature extractors, transformers, and estimators inside `sklearn.pipeline.Pipeline` or equivalent frameworks to guarantee transformers only call `.fit()` on training folds.

---


In [2]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Set random seed for reproducibility
np.random.seed(42)

# Generate pure noise features with zero true relationship to target
n_samples = 200
n_features = 1000
X = np.random.randn(n_samples, n_features)
y = np.random.choice([0, 1], size=n_samples)

print("--- DEMONSTRATION OF DATA LEAKAGE ---")

# =====================================================================
# 1. THE LEAKY APPROACH: Feature selection applied to ALL data first
# =====================================================================
# Selecting top 20 features before splitting
leaky_selector = SelectKBest(score_func=f_classif, k=20)
X_leaky_selected = leaky_selector.fit_transform(X, y)

# Split the already-filtered data
X_train_leak, X_test_leak, y_train_leak, y_test_leak = train_test_split(
    X_leaky_selected, y, test_size=0.3, random_state=42
)

leaky_model = LogisticRegression()
leaky_model.fit(X_train_leak, y_train_leak)
leaky_train_acc = leaky_model.score(X_train_leak, y_train_leak)
leaky_test_acc = leaky_model.score(X_test_leak, y_test_leak)

print(f"Leaky Pipeline -> Train Acc: {leaky_train_acc:.3f} | Test Acc: {leaky_test_acc:.3f}")
# Notice: Test accuracy will artificially hover around 70-85% on pure noise!

# =====================================================================
# 2. THE SOUND APPROACH: Encapsulated Scikit-Learn Pipeline
# =====================================================================
# Split raw data first
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Enclose transformations strictly inside the pipeline
proper_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('selector', SelectKBest(score_func=f_classif, k=20)),
    ('classifier', LogisticRegression())
])

proper_pipeline.fit(X_train, y_train)
clean_train_acc = proper_pipeline.score(X_train, y_train)
clean_test_acc = proper_pipeline.score(X_test, y_test)

print(f"Sound Pipeline -> Train Acc: {clean_train_acc:.3f} | Test Acc: {clean_test_acc:.3f}")
# Notice: Test accuracy drops to the expected ~50% random chance baseline.


--- DEMONSTRATION OF DATA LEAKAGE ---
Leaky Pipeline -> Train Acc: 0.864 | Test Acc: 0.717
Sound Pipeline -> Train Acc: 0.850 | Test Acc: 0.450
